# UCF-Crime R3D-18 violence baseline

This is the portable project copy of the supplied notebook. It trains and evaluates a Fighting-versus-Normal R3D-18 baseline on the UCF-Crime subset listed in the metadata. The raw dataset is not included.

Set UCF_CRIME_ROOT to the dataset folder before launching Jupyter, or place the class folders and metadata under data/behavior/raw/UCF-Crime/. Extracted clips and checkpoints are stored under this project. Rerunning clip extraction reuses the output folder and does not delete existing files.

Keep the official video-level split manifest unchanged so clips from one source video cannot cross train, validation, and test splits. The test set should only be used after model selection.


In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. Check Runtime settings.")

In [ ]:
# Portable project and UCF-Crime dataset paths.
# Set UCF_CRIME_ROOT if your downloaded dataset is stored elsewhere.
import os
from pathlib import Path

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "src" / "behavior" / "train.py").is_file()),
    Path.cwd(),
)
PROJECT = Path(os.environ.get("CAMPUS_THREAT_ROOT", str(PROJECT_ROOT))).resolve()
DATA_ROOT = Path(
    os.environ.get("UCF_CRIME_ROOT", str(PROJECT / "data" / "behavior" / "raw" / "UCF-Crime"))
).resolve()

FIGHTING_DIR = DATA_ROOT / "Fighting"
NORMAL_DIR = DATA_ROOT / "Normal_Videos_for_Event_Recognition"
METADATA_DIR = DATA_ROOT / "metadata"
OUTPUT_DIR = PROJECT / "data" / "behavior" / "processed" / "ucf_crime"
MODEL_DIR = PROJECT / "models" / "behavior" / "ucf_crime"

print("Project root:", PROJECT)
print("UCF-Crime root:", DATA_ROOT)
print("Dataset found:", DATA_ROOT.exists())

In [ ]:
from pathlib import Path

for required_path in [FIGHTING_DIR, NORMAL_DIR, METADATA_DIR]:
    if not required_path.exists():
        raise FileNotFoundError(
            f"Missing UCF-Crime folder: {required_path}. "
            "Set UCF_CRIME_ROOT to the folder containing the class folders and metadata."
        )

print("Fighting videos:", len(list(FIGHTING_DIR.glob("*.mp4"))))
print("Normal videos:", len(list(NORMAL_DIR.glob("*.mp4"))))
print("Metadata files:")
for file in sorted(METADATA_DIR.iterdir()):
    print(file.name)

In [ ]:
import pandas as pd

manifest = pd.read_csv(METADATA_DIR / "split_manifest.csv")
intervals = pd.read_csv(METADATA_DIR / "fighting_intervals.csv")

print("========== MANIFEST ==========")

print("\nColumn names:")
print(manifest.columns.tolist())

print("\nFirst 5 rows:")
display(manifest.head())

print("\nVideos by split and category:")
print(pd.crosstab(manifest["split"], manifest["category"]))

print("\n========== FIGHTING INTERVALS ==========")

print("\nColumn names:")
print(intervals.columns.tolist())

print("\nFirst 5 rows:")
display(intervals.head())

print("\nIntervals by split:")
print(intervals.groupby("split").size())

In [ ]:
from pathlib import Path

import cv2
import pandas as pd

manifest = pd.read_csv(METADATA_DIR / "split_manifest.csv")
intervals = pd.read_csv(METADATA_DIR / "fighting_intervals.csv")


# Map each category to its Google Drive folder
def get_video_path(row):
    if row["category"] == "Fighting":
        return FIGHTING_DIR / row["filename"]
    elif row["category"] == "Normal_Videos_event":
        return NORMAL_DIR / row["filename"]
    else:
        return None


manifest["drive_path"] = manifest.apply(get_video_path, axis=1)

print("Missing videos:")
missing = manifest[
    ~manifest["drive_path"].apply(lambda p: p is not None and p.exists())
]
print(missing[["filename", "category"]])

print("\nTotal missing:", len(missing))

# Validate fighting interval durations
intervals["duration_seconds"] = intervals["end_seconds"] - intervals["start_seconds"]

print("\nFighting interval duration statistics:")
print(intervals["duration_seconds"].describe())

print("\nIntervals shorter than 2 seconds:", (intervals["duration_seconds"] < 2).sum())

print("\nIntervals at least 2 seconds:", (intervals["duration_seconds"] >= 2).sum())

In [ ]:
!pip install -q opencv-python-headless pandas numpy tqdm

In [ ]:
import random
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# -----------------------------
# CONFIGURATION
# -----------------------------

SEED = 42
FRAMES_PER_CLIP = 16
SAMPLE_FPS = 8
CLIP_DURATION = 2.0
IMAGE_SIZE = 112
MAX_CLIPS_PER_VIDEO = 3

random.seed(SEED)
np.random.seed(SEED)

# Save extracted clips temporarily on Colab storage
OUTPUT_DIR = PROJECT / "data" / "behavior" / "processed" / "ucf_crime"

# Reuse the output folder. Existing data is not deleted on rerun.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

manifest = pd.read_csv(METADATA_DIR / "split_manifest.csv")
intervals = pd.read_csv(METADATA_DIR / "fighting_intervals.csv")

# -----------------------------
# VIDEO PATH MAPPING
# -----------------------------


def get_video_path(row):

    if row["category"] == "Fighting":
        return FIGHTING_DIR / row["filename"]

    elif row["category"] == "Normal_Videos_event":
        return NORMAL_DIR / row["filename"]

    raise ValueError(f"Unknown category: {row['category']}")


manifest["drive_path"] = manifest.apply(get_video_path, axis=1)

# -----------------------------
# EXTRACT ONE CLIP
# -----------------------------


def extract_clip(cap, start_seconds, fps):

    frame_indices = [
        int(round((start_seconds + i / SAMPLE_FPS) * fps))
        for i in range(FRAMES_PER_CLIP)
    ]

    frames = []

    for frame_idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ret, frame = cap.read()

        if not ret:
            return None

        frame = cv2.resize(frame, (IMAGE_SIZE, IMAGE_SIZE))

        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(frame)

    return np.stack(frames).astype(np.uint8)


# -----------------------------
# BUILD FIGHTING CLIP CANDIDATES
# -----------------------------


def get_fighting_starts(video_intervals):

    candidates = []

    for _, interval in video_intervals.iterrows():
        start = float(interval["start_seconds"])
        end = float(interval["end_seconds"])

        latest_start = end - CLIP_DURATION

        if latest_start < start:
            continue

        # Non-overlapping candidate windows
        current = start

        while current <= latest_start + 1e-6:
            candidates.append(round(current, 4))
            current += CLIP_DURATION

    # Select up to 3 evenly distributed candidate windows
    if len(candidates) > MAX_CLIPS_PER_VIDEO:
        selected_indices = np.linspace(
            0, len(candidates) - 1, MAX_CLIPS_PER_VIDEO, dtype=int
        )

        candidates = [candidates[i] for i in selected_indices]

    return candidates


# -----------------------------
# BUILD NORMAL CLIP CANDIDATES
# -----------------------------


def get_normal_starts(duration):

    latest_start = duration - CLIP_DURATION

    if latest_start <= 0:
        return []

    # Sample from different parts of the video
    fractions = [0.15, 0.50, 0.80]

    return [round(f * latest_start, 4) for f in fractions[:MAX_CLIPS_PER_VIDEO]]


# -----------------------------
# EXTRACT ALL CLIPS
# -----------------------------

records = []

for _, row in tqdm(manifest.iterrows(), total=len(manifest), desc="Processing videos"):
    video_path = Path(row["drive_path"])

    cap = cv2.VideoCapture(str(video_path))

    if not cap.isOpened():
        print("Could not open:", video_path)
        continue

    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_count = cap.get(cv2.CAP_PROP_FRAME_COUNT)

    if fps <= 0 or frame_count <= 0:
        cap.release()
        continue

    duration = frame_count / fps

    if row["category"] == "Fighting":
        video_intervals = intervals[intervals["video_id"] == row["original_video_id"]]

        starts = get_fighting_starts(video_intervals)
        label = 1
        class_name = "VIOLENCE"

    else:
        starts = get_normal_starts(duration)
        label = 0
        class_name = "NON_VIOLENCE"

    for clip_number, start_seconds in enumerate(starts):
        # Make sure the clip is inside the source video
        if start_seconds + CLIP_DURATION > duration:
            continue

        clip = extract_clip(cap, start_seconds, fps)

        if clip is None:
            continue

        clip_id = (
            f"{row['split']}_{row['original_video_id']}_{class_name}_{clip_number:02d}"
        )

        clip_id = clip_id.replace("/", "_").replace("\\", "_")

        save_dir = OUTPUT_DIR / row["split"] / class_name
        save_dir.mkdir(parents=True, exist_ok=True)

        save_path = save_dir / f"{clip_id}.npy"

        np.save(save_path, clip)

        records.append(
            {
                "clip_id": clip_id,
                "clip_path": str(save_path),
                "video_id": row["original_video_id"],
                "filename": row["filename"],
                "split": row["split"],
                "label": label,
                "class_name": class_name,
                "start_seconds": start_seconds,
                "end_seconds": start_seconds + CLIP_DURATION,
            }
        )

    cap.release()


# -----------------------------
# CREATE CLIP METADATA
# -----------------------------

clip_df = pd.DataFrame(records)

if clip_df.empty:
    raise RuntimeError("No clips extracted. Check video paths and annotations.")

# Balance the classes separately within each split
balanced_parts = []

for split in ["train", "validation", "test"]:
    part = clip_df[clip_df["split"] == split]

    counts = part["label"].value_counts()

    if 0 not in counts or 1 not in counts:
        raise RuntimeError(f"Missing one class in {split}. Counts: {counts.to_dict()}")

    target_count = min(counts[0], counts[1])

    for label in [0, 1]:
        class_part = part[part["label"] == label]

        # Deterministic undersampling
        class_part = class_part.sample(n=target_count, random_state=SEED)

        balanced_parts.append(class_part)

balanced_df = pd.concat(balanced_parts, ignore_index=True)

balanced_df.to_csv(OUTPUT_DIR / "clip_metadata.csv", index=False)

# -----------------------------
# RESULTS
# -----------------------------

print("\n========== EXTRACTION COMPLETE ==========")

print("Total clips before balancing:", len(clip_df))
print("Total clips after balancing:", len(balanced_df))

print("\nBalanced clips by split and class:")

print(pd.crosstab(balanced_df["split"], balanced_df["class_name"]))

print("\nClip shape:", np.load(balanced_df.iloc[0]["clip_path"]).shape)

print("\nMetadata saved at:", OUTPUT_DIR / "clip_metadata.csv")
print("Clips saved at:", OUTPUT_DIR)

In [ ]:
clip_df = pd.read_csv(OUTPUT_DIR / "clip_metadata.csv")

print("Total clips:", len(clip_df))

print("\nClass distribution:")
print(clip_df["class_name"].value_counts())

print("\nSplit distribution:")
print(pd.crosstab(clip_df["split"], clip_df["class_name"]))

print("\nUnique source videos per split:")
print(clip_df.groupby(["split", "class_name"])["video_id"].nunique())

print("\nMissing clip files:")
print(sum(not Path(p).exists() for p in clip_df["clip_path"]))

In [ ]:
# SETTING ENVIRONMENT FOR TRAINING
import os

import numpy as np
import pandas as pd
import torch
import torchvision

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("GPU available:", torch.cuda.is_available())

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
from pathlib import Path

import torch
from torch.utils.data import DataLoader, Dataset

clip_df = pd.read_csv(OUTPUT_DIR / "clip_metadata.csv")


class ViolenceDataset(Dataset):
    def __init__(self, dataframe):
        self.data = dataframe.reset_index(drop=True)

        self.mean = torch.tensor([0.43216, 0.394666, 0.37645]).view(3, 1, 1, 1)

        self.std = torch.tensor([0.22803, 0.22145, 0.216989]).view(3, 1, 1, 1)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):

        row = self.data.iloc[index]

        frames = np.load(row["clip_path"])

        # [T,H,W,C] -> [C,T,H,W]
        frames = torch.from_numpy(frames).permute(3, 0, 1, 2)
        frames = frames.float() / 255.0

        frames = (frames - self.mean) / self.std

        label = torch.tensor(int(row["label"]), dtype=torch.long)

        return frames, label


train_df = clip_df[clip_df["split"] == "train"]
val_df = clip_df[clip_df["split"] == "validation"]

train_dataset = ViolenceDataset(train_df)
val_dataset = ViolenceDataset(val_df)

train_loader = DataLoader(
    train_dataset, batch_size=4, shuffle=True, num_workers=2, pin_memory=True
)

val_loader = DataLoader(
    val_dataset, batch_size=4, shuffle=False, num_workers=2, pin_memory=True
)

# Check one batch
videos, labels = next(iter(train_loader))

print("Video batch shape:", videos.shape)
print("Label batch shape:", labels.shape)
print("Labels:", labels)

In [ ]:
import torch
import torch.nn as nn
from torchvision.models.video import R3D_18_Weights, r3d_18

# Load pretrained Kinetics-400 model
weights = R3D_18_Weights.KINETICS400_V1

model = r3d_18(weights=weights)

# Freeze pretrained backbone
for param in model.parameters():
    param.requires_grad = False

# Replace final layer for binary classification
model.fc = nn.Linear(model.fc.in_features, 2)

# Move model to GPU
model = model.to(device)

print("R3D-18 loaded successfully!")
print("Final classification layer:", model.fc)
print("Device:", next(model.parameters()).device)

In [ ]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(model.fc.parameters(), lr=0.001, weight_decay=0.01)

EPOCHS = 8

best_val_accuracy = 0.0
patience = 3
epochs_without_improvement = 0

MODEL_DIR = PROJECT / "models" / "behavior" / "ucf_crime"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

best_model_path = MODEL_DIR / "r3d18_best.pth"

history = []

for epoch in range(EPOCHS):
    # -------------------
    # TRAINING
    # -------------------

    model.eval()
    model.fc.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for videos, labels in train_loader:
        videos = videos.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()

        outputs = model(videos)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * labels.size(0)

        predictions = outputs.argmax(dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    train_loss = running_loss / total
    train_accuracy = correct / total

    # -------------------
    # VALIDATION
    # -------------------

    model.eval()

    val_loss_total = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for videos, labels in val_loader:
            videos = videos.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model(videos)
            loss = criterion(outputs, labels)

            val_loss_total += loss.item() * labels.size(0)

            predictions = outputs.argmax(dim=1)

            val_correct += (predictions == labels).sum().item()
            val_total += labels.size(0)

    val_loss = val_loss_total / val_total
    val_accuracy = val_correct / val_total

    history.append(
        {
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "train_accuracy": train_accuracy,
            "val_loss": val_loss,
            "val_accuracy": val_accuracy,
        }
    )

    print(f"\nEpoch {epoch + 1}/{EPOCHS}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Train Accuracy: {train_accuracy:.4f}")
    print(f"Validation Loss: {val_loss:.4f}")
    print(f"Validation Accuracy: {val_accuracy:.4f}")

    # Save best model
    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy

        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "best_val_accuracy": best_val_accuracy,
                "epoch": epoch + 1,
                "class_names": ["NON_VIOLENCE", "VIOLENCE"],
            },
            best_model_path,
        )

        epochs_without_improvement = 0

        print("Best model checkpoint saved.")

    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= patience:
        print("\nEarly stopping triggered.")
        break

print("\nTraining completed.")
print("Best validation accuracy:", best_val_accuracy)
print("Checkpoint:", best_model_path)

In [ ]:
# Create test dataloader
from torch.utils.data import DataLoader

test_df = clip_df[clip_df["split"] == "test"]

test_dataset = ViolenceDataset(test_df)

test_loader = DataLoader(
    test_dataset, batch_size=4, shuffle=False, num_workers=2, pin_memory=True
)

print("Test clips:", len(test_dataset))
print(test_df["class_name"].value_counts())

In [ ]:
import torch

checkpoint = torch.load(best_model_path, map_location=device, weights_only=True)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("Best checkpoint loaded.")
print("Best validation accuracy:", checkpoint["best_val_accuracy"])
print("Best epoch:", checkpoint["epoch"])

In [ ]:
import numpy as np

all_labels = []
all_predictions = []
all_probabilities = []

model.eval()

with torch.no_grad():
    for videos, labels in test_loader:
        videos = videos.to(device)

        outputs = model(videos)

        probabilities = torch.softmax(outputs, dim=1)

        predictions = torch.argmax(outputs, dim=1)

        all_labels.extend(labels.numpy())
        all_predictions.extend(predictions.cpu().numpy())
        all_probabilities.extend(probabilities[:, 1].cpu().numpy())

all_labels = np.array(all_labels)
all_predictions = np.array(all_predictions)
all_probabilities = np.array(all_probabilities)

print("Test predictions completed.")

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
)

print("TEST ACCURACY:", accuracy_score(all_labels, all_predictions))

print("\nCLASSIFICATION REPORT:\n")

print(
    classification_report(
        all_labels,
        all_predictions,
        target_names=["NON_VIOLENCE", "VIOLENCE"],
        zero_division=0,
    )
)

cm = confusion_matrix(all_labels, all_predictions, labels=[0, 1])

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm, display_labels=["NON_VIOLENCE", "VIOLENCE"]
)

disp.plot(values_format="d")
plt.title("R3D-18 Test Confusion Matrix")
plt.show()

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix

# Create prediction dataframe
results = test_df.reset_index(drop=True).copy()

results["actual"] = all_labels
results["predicted"] = all_predictions
results["probability"] = all_probabilities

# Aggregate predictions by source video
video_results = (
    results.groupby(["video_id", "label"])
    .agg(
        total_clips=("predicted", "count"),
        violence_predictions=("predicted", "sum"),
        violence_probability=("probability", "mean"),
    )
    .reset_index()
)

# Majority voting
video_results["video_prediction"] = (
    video_results["violence_predictions"] > video_results["total_clips"] / 2
).astype(int)

print("VIDEO-LEVEL RESULTS")
print(video_results)

print("\nVIDEO-LEVEL CLASSIFICATION REPORT")

print(
    classification_report(
        video_results["label"],
        video_results["video_prediction"],
        labels=[0, 1],
        target_names=["NON_VIOLENCE", "VIOLENCE"],
        zero_division=0,
    )
)

print("\nVIDEO-LEVEL CONFUSION MATRIX")

print(
    confusion_matrix(
        video_results["label"], video_results["video_prediction"], labels=[0, 1]
    )
)

In [ ]:
# ROLLING WINDOW VIOLENCE DETECTION
import torch
import torch.nn as nn
from torchvision.models.video import r3d_18

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

checkpoint = torch.load(
    PROJECT / "models" / "behavior" / "ucf_crime" / "r3d18_best.pth",
    map_location=device,
    weights_only=True,
)

model = r3d_18(weights=None)
model.fc = nn.Linear(model.fc.in_features, 2)

model.load_state_dict(checkpoint["model_state_dict"])
model = model.to(device)
model.eval()

print("Best R3D-18 model loaded.")
print("Device:", device)
print("Best validation accuracy:", checkpoint["best_val_accuracy"])

In [ ]:
# Create the rolling-window inference function
from collections import deque

import cv2
import numpy as np
import torch

# Same normalization used during training
MEAN = torch.tensor([0.43216, 0.394666, 0.37645]).view(1, 3, 1, 1, 1).to(device)

STD = torch.tensor([0.22803, 0.22145, 0.216989]).view(1, 3, 1, 1, 1).to(device)


def predict_window(frames):

    # frames: list of 16 RGB frames
    video = np.stack(frames)

    # [T,H,W,C] -> [1,C,T,H,W]
    video = torch.from_numpy(video).permute(3, 0, 1, 2)
    video = video.unsqueeze(0).float().to(device) / 255.0

    video = (video - MEAN) / STD

    with torch.no_grad():
        outputs = model(video)
        probabilities = torch.softmax(outputs, dim=1)

    violence_probability = probabilities[0, 1].item()

    return violence_probability


def detect_violence_video(video_path, threshold=0.5):

    cap = cv2.VideoCapture(str(video_path))

    if not cap.isOpened():
        raise RuntimeError(f"Cannot open video: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS)

    if fps <= 0:
        cap.release()
        raise RuntimeError("Invalid video FPS.")

    # Sample approximately 8 frames per second
    frame_step = max(1, round(fps / 8))

    frame_buffer = deque(maxlen=16)
    probability_buffer = deque(maxlen=3)

    frame_index = 0
    predictions = []

    while True:
        ret, frame = cap.read()

        if not ret:
            break

        if frame_index % frame_step == 0:
            frame = cv2.resize(frame, (112, 112))
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

            frame_buffer.append(frame)

            # Predict when 16 frames are available
            if len(frame_buffer) == 16:
                probability = predict_window(list(frame_buffer))

                probability_buffer.append(probability)

                smoothed_probability = float(np.mean(probability_buffer))

                label = (
                    "VIOLENCE" if smoothed_probability >= threshold else "NON_VIOLENCE"
                )

                timestamp = frame_index / fps

                predictions.append(
                    {
                        "timestamp": round(timestamp, 2),
                        "violence_probability": round(probability, 4),
                        "smoothed_probability": round(smoothed_probability, 4),
                        "prediction": label,
                    }
                )

                print(
                    f"Time: {timestamp:.2f}s | "
                    f"Violence probability: {smoothed_probability:.3f} | "
                    f"Prediction: {label}"
                )

        frame_index += 1

    cap.release()

    return predictions

In [ ]:
test_video = FIGHTING_DIR / "Fighting003_x264.mp4"

predictions = detect_violence_video(test_video)

print("\nTotal prediction windows:", len(predictions))

if predictions:
    violence_count = sum(p["prediction"] == "VIOLENCE" for p in predictions)

    print("Violence windows:", violence_count)
    print(
        "Percentage of windows flagged:",
        round(100 * violence_count / len(predictions), 2),
        "%",
    )

In [ ]:
# testing on a normal video
normal_test_video = NORMAL_DIR / "Normal_Videos_015_x264.mp4"

normal_predictions = detect_violence_video(normal_test_video)

print("\n========== NORMAL VIDEO RESULTS ==========")

total = len(normal_predictions)

violence_count = sum(p["prediction"] == "VIOLENCE" for p in normal_predictions)

normal_count = total - violence_count

print("Total windows:", total)
print("Violence windows:", violence_count)
print("Normal windows:", normal_count)

if total > 0:
    print("False alarm window percentage:", round(violence_count / total * 100, 2), "%")

In [ ]:
filename = "Normal_Videos_015_x264.mp4"

print(manifest[manifest["filename"] == filename][["filename", "split", "category"]])

In [ ]:
validation_videos = manifest[manifest["split"] == "validation"]

print(
    validation_videos[["original_video_id", "filename", "category"]].to_string(
        index=False
    )
)

In [ ]:
# generating the pt file
from pathlib import Path

import torch

MODEL_DIR = PROJECT / "models" / "behavior" / "ucf_crime"

checkpoint = torch.load(
    MODEL_DIR / "r3d18_best.pth", map_location="cpu", weights_only=True
)

pt_path = MODEL_DIR / "violence_detection_r3d18.pt"

torch.save(
    {
        "model_state_dict": checkpoint["model_state_dict"],
        "model_name": "R3D-18",
        "num_classes": 2,
        "class_names": ["NON_VIOLENCE", "VIOLENCE"],
        "input_shape": [3, 16, 112, 112],
        "best_validation_accuracy": checkpoint["best_val_accuracy"],
        "best_epoch": checkpoint["epoch"],
    },
    pt_path,
)

print("Export successful!")
print("File:", pt_path)
print("Size:", round(pt_path.stat().st_size / (1024**2), 2), "MB")